<a href="https://colab.research.google.com/github/shreyansh8h/hello-world/blob/master/June6_Shreyans_gemini_dynamic_prompt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
!pip install --upgrade langchain langchain-google-genai langchain-community

In [20]:
# check langchain version if not uncomment below pip command to install the correct version
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)
# The following line is commented out as the installation is handled in another cell.
# !pip install --upgrade langchain langchain-google-genai langchain-community

1.3.4
0.4.2


In [29]:
import os
import json
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from textwrap import indent
from dotenv import load_dotenv
from google.colab import userdata

# Load environment variables
load_dotenv()

import warnings
warnings.filterwarnings('ignore')

# Configure llm via .env or Colab secrets
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0, google_api_key=GOOGLE_API_KEY)

# Define a dynamic routing ruleset


In [30]:
ROUTING_RULES = """
1️⃣ If CUSTOMER_TIER = "Enterprise" AND SENTIMENT_SCORE < 0.3:
    Priority = "URGENT"
    Route to = "Senior Support"
    SLA_TARGET = "1 hour"
    ESCALATION_CRITERIA = "If not acknowledged in 30 minutes"

2️⃣ If PRODUCT_AREA = "Billing" AND EXTRACTED_KEYWORDS contain any of ["refund", "charge", "payment"]:
    Route to = "Billing Team"
    SLA_TARGET = "4 hours"
    TEMPLATE_ID = "Billing_Issue_Response"

3️⃣ If EXTRACTED_KEYWORDS contain any of ["bug", "error", "broken"] AND CUSTOMER_TIER = "Premium":
    Priority = "HIGH"
    Route to = "Technical Team"
    SLA_TARGET = "2 hours"
    TEMPLATE_ID = "Tech_Error_Response"

4️⃣ Default rule:
    Priority = "MEDIUM"
    Route to = "General Support"
    SLA_TARGET = "6 hours"
    TEMPLATE_ID = "Generic_Response"

    """

In [31]:
# Example input values — you can replace these dynamically
ticket_input = {
    "CUSTOMER_TIER": "Enterprise",
    "PRODUCT_AREA": "Billing",
    "SENTIMENT_SCORE": 0.25,
    "EXTRACTED_KEYWORDS": ["refund", "delay"]
}

In [32]:
# Build the dynamic prompt
prompt = f"""
You are an AI assistant for a Customer Support Ticket Analysis System.
Your task is to analyze the ticket details and determine routing priority,
SLA, escalation, and response template based on the rules.

[INPUTS]
Customer Tier: {ticket_input['CUSTOMER_TIER']}
Product Area: {ticket_input['PRODUCT_AREA']}
Sentiment Score: {ticket_input['SENTIMENT_SCORE']}
Extracted Keywords: {ticket_input['EXTRACTED_KEYWORDS']}

[ROUTING RULES]
{ROUTING_RULES}

[OUTPUT FORMAT]
Priority: <CALCULATED_PRIORITY>
Route to: <ASSIGNED_TEAM>
Suggested response time: <SLA_TARGET>
Initial response template: <TEMPLATE_ID>
Escalation trigger: <ESCALATION_CRITERIA>

Now output only the structured result, no explanation.
"""

In [25]:
#call the model



In [33]:
# --- TUNED PROMPT RESPONSE ---
dynamic_messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=prompt)
]
dynamic_response = llm.invoke(dynamic_messages).content.strip()

In [34]:
print("Dynamic PROMPT:")
print(indent(dynamic_response, "    "))

Dynamic PROMPT:
    Priority: URGENT
    Route to: Billing Team
    Suggested response time: 4 hours
    Initial response template: Billing_Issue_Response
    Escalation trigger: If not acknowledged in 30 minutes


# To Do: Sample ticket_input test cases to try One by one test and observe output


In [35]:
#Test Case 1 — Enterprise + Low Sentiment (Rule 1)
ticket_input = {
    "CUSTOMER_TIER": "Enterprise",
    "PRODUCT_AREA": "App",
    "SENTIMENT_SCORE": 0.20,
    "EXTRACTED_KEYWORDS": ["slow", "crash"]
}
#Test Case 2 — Billing + Refund Keyword (Rule 2)
ticket_input = {
    "CUSTOMER_TIER": "Standard",
    "PRODUCT_AREA": "Billing",
    "SENTIMENT_SCORE": 0.75,
    "EXTRACTED_KEYWORDS": ["refund"]
}
#Test Case 3 — Billing but No Keyword (Default Rule)
ticket_input = {
    "CUSTOMER_TIER": "Standard",
    "PRODUCT_AREA": "Billing",
    "SENTIMENT_SCORE": 0.60,
    "EXTRACTED_KEYWORDS": ["question", "invoice"]
}
#Test Case 4 — Premium + Bug Keyword (Rule 3)
ticket_input = {
    "CUSTOMER_TIER": "Premium",
    "PRODUCT_AREA": "App",
    "SENTIMENT_SCORE": 0.55,
    "EXTRACTED_KEYWORDS": ["bug", "login failure"]
}
#Test Case 5 — Premium + No Bug Keyword (Default Rule)
ticket_input = {
    "CUSTOMER_TIER": "Enterprise",
    "PRODUCT_AREA": "Billing",
    "SENTIMENT_SCORE": 0.85,
    "EXTRACTED_KEYWORDS": ["question"]
}
#Test Case 6 — Enterprise but Good Sentiment (Default Rule)
ticket_input = {
    "CUSTOMER_TIER": "Enterprise",
    "PRODUCT_AREA": "Billing",
    "SENTIMENT_SCORE": 0.85,
    "EXTRACTED_KEYWORDS": ["question"]
}
#Test Case 7 — Multi-Trigger (Billing + Refund) — Should Use Rule 2
ticket_input = {
    "CUSTOMER_TIER": "Premium",
    "PRODUCT_AREA": "Billing",
    "SENTIMENT_SCORE": 0.40,
    "EXTRACTED_KEYWORDS": ["refund", "bug"]
}
#Test Case 9 — Enterprise + Bug + High Sentiment — Rule 3 Does NOT apply because tier ≠ Premium → Default Rule
ticket_input = {
    "CUSTOMER_TIER": "Enterprise",
    "PRODUCT_AREA": "App",
    "SENTIMENT_SCORE": 0.90,
    "EXTRACTED_KEYWORDS": ["bug"]
}
#Test Case 10 — Billing + Charge Keyword (Rule 2)
ticket_input = {
    "CUSTOMER_TIER": "Standard",
    "PRODUCT_AREA": "Billing",
    "SENTIMENT_SCORE": 0.33,
    "EXTRACTED_KEYWORDS": ["charge", "payment"]
}


# Task
The user is attempting to use a LangChain `ChatGoogleGenerativeAI` model to analyze customer support ticket details and determine routing priority, SLA, escalation, and response template based on a predefined set of rules. The initial attempt resulted in a `ChatGoogleGenerativeAIError` indicating that the model 'gemini-flash' was not found or supported.

The overall goal is to fix this error by updating the model name to a correct, available model and then re-running the LLM invocation to successfully generate a dynamic response based on the provided routing rules and ticket input. After fixing the issue, the task involves displaying the generated response and confirming the correct integration and functionality of the Gemini Flash 2.5 model.

## Update Gemini Model Name

### Subtask:
Modify the `llm` initialization to use the correct model name `gemini-1.5-flash`.


**Reasoning**:
I need to execute the cell `a28bafbc` to re-initialize the `llm` object with the updated model name.



In [36]:
import os
import json
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from textwrap import indent
from dotenv import load_dotenv
from google.colab import userdata

# Load environment variables
load_dotenv()

import warnings
warnings.filterwarnings('ignore')

# Configure llm via .env or Colab secrets
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0, google_api_key=GOOGLE_API_KEY)


## Re-initialize LLM

### Subtask:
Execute the cell that initializes the `llm` object to incorporate the corrected model name.


**Reasoning**:
Executing the cell `a28bafbc` will re-initialize the `llm` object with the `gemini-1.5-flash` model, which was the objective of the previous subtask and this subtask.



In [37]:
import os
import json
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from textwrap import indent
from dotenv import load_dotenv
from google.colab import userdata

# Load environment variables
load_dotenv()

import warnings
warnings.filterwarnings('ignore')

# Configure llm via .env or Colab secrets
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0, google_api_key=GOOGLE_API_KEY)

## Re-run LLM Invocation

### Subtask:
Re-invoke the LLM with the corrected model to process the ticket details and routing rules.


**Reasoning**:
I need to re-run the LLM invocation in cell `f852c6df` to generate the dynamic response using the correctly initialized `llm` object.



In [38]:
dynamic_messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=prompt)
]
dynamic_response = llm.invoke(dynamic_messages).content.strip()

### Verify Model Availability

To verify which models are available and supported in your environment, you can list all models accessible via your API key. This will confirm if `gemini-2.5-flash` (or any other Gemini model) is indeed available.

After running the above cell, please check the output to confirm if `gemini-2.5-flash` is listed among the available models with `generateContent` as a supported method. If it's not listed, or if a different model name is preferred, you might need to adjust the `llm` initialization accordingly.